<!-- NB_DOC_INTRO_V1 -->
# Structures de donnees — Lists, Tuples, Dicts, Sets, NumPy

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Cheat-sheet des **structures de donnees built-in Python** (list, tuple, dict, set) et de **NumPy arrays**. Patterns courants en data science : flatten de listes imbriquees, merge de dicts, decoupage en N parties equilibrees, manipulations numpy (broadcast, reshape, masks).

Pour pandas : voir [Structures_DataFrame.ipynb](./Structures_DataFrame.ipynb).

## Plan

1. Lists (slicing, comprehensions, perf)
2. Tuples (named tuples, immutabilite)
3. Dicts (merge, subset, defaultdict, Counter)
4. Sets (operations ensemblistes)
5. Patterns avances (flatten, decoupage N parts)
6. NumPy arrays (creation, reshape, broadcasting, masks)
7. Comparatif perf : list vs tuple vs set vs dict
8. References


In [ ]:
import numpy as np
from collections import defaultdict, Counter, OrderedDict
from itertools import chain, groupby
import sys
import time

## 1. Lists — slicing, comprehensions

In [ ]:
xs = list(range(10))
print(f"xs = {xs}")

# Slicing : [start:stop:step]
print(f"xs[2:7]     = {xs[2:7]}")
print(f"xs[::2]     = {xs[::2]}     # 1 sur 2")
print(f"xs[::-1]    = {xs[::-1]}    # reverse")
print(f"xs[-3:]     = {xs[-3:]}")

# Comprehensions (preferable a map + lambda)
squared = [x**2 for x in xs if x % 2 == 0]
print(f"\nsquared evens = {squared}")

# Nested comprehension
grid = [[i*j for j in range(5)] for i in range(3)]
print(f"\ngrid = {grid}")

# Methodes courantes
xs.append(100); xs.extend([200, 300]); xs.insert(0, -1); xs.remove(5); xs.pop(3)
print(f"\nApres modifications : {xs}")

## 2. Tuples — immutable, named tuples

In [ ]:
# Tuple basique : immutable
t = (1, 2, 3)
# t[0] = 99  # TypeError !

# Unpacking
a, b, c = t
print(f"unpack : a={a} b={b} c={c}")

# Star unpacking
first, *middle, last = (1, 2, 3, 4, 5)
print(f"first={first}, middle={middle}, last={last}")

# === Named tuples (lisibilite + immutabilite) ===
from collections import namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(1, 2)
print(f"\np = {p}, p.x={p.x}, p.y={p.y}")
print(f"asdict : {p._asdict()}")

# === Mieux : dataclass (Python 3.7+) ===
from dataclasses import dataclass

@dataclass(frozen=True)              # frozen = immutable
class PointDC:
    x: float
    y: float
    def distance_to(self, other: "PointDC") -> float:
        return ((self.x - other.x)**2 + (self.y - other.y)**2)**0.5

p1 = PointDC(0, 0); p2 = PointDC(3, 4)
print(f"\ndistance : {p1.distance_to(p2)}")

## 3. Dicts — merge, subset, defaultdict, Counter

In [ ]:
d1 = {"a": 1, "b": 2}
d2 = {"b": 99, "c": 3}

# === Merge (Python 3.9+) ===
merged = d1 | d2                          # d2 ecrase d1
print(f"merge       : {merged}")

# Update in-place
d1 |= d2
print(f"in-place    : {d1}")

# === Subset (extract keys) ===
data = {"a": 1, "b": 2, "c": 3, "d": 4}
keys_wanted = ["a", "c"]
subset = {k: data[k] for k in keys_wanted if k in data}
print(f"\nsubset      : {subset}")

# === defaultdict : evite KeyError ===
counts = defaultdict(int)
for word in ["the", "quick", "the", "fox", "the", "fox"]:
    counts[word] += 1
print(f"\ncounts      : {dict(counts)}")

# === Counter (specialise) ===
c = Counter(["the", "quick", "the", "fox", "the", "fox"])
print(f"Counter most_common(2) : {c.most_common(2)}")

# === Parcours ===
for k, v in data.items():
    print(f"  {k} -> {v}")

## 4. Sets — operations ensemblistes

In [ ]:
s1 = {1, 2, 3, 4}
s2 = {3, 4, 5, 6}

print(f"intersection : {s1 & s2}")
print(f"union        : {s1 | s2}")
print(f"diff s1-s2   : {s1 - s2}")
print(f"symmetric    : {s1 ^ s2}")          # XOR

# Frozenset (immutable)
fs = frozenset([1, 2, 3])
# fs.add(4)  # AttributeError

# Membership : O(1) (dict-like)
big_list = list(range(10_000))
big_set = set(big_list)
target = 9999
print(f"\nLookup membership (10k items) :")
t0 = time.perf_counter(); _ = target in big_list; t_list = time.perf_counter() - t0
t0 = time.perf_counter(); _ = target in big_set; t_set = time.perf_counter() - t0
print(f"  list : {t_list*1e6:.2f} us")
print(f"  set  : {t_set*1e6:.2f} us  ({t_list/t_set:.0f}x plus rapide)")

## 5. Patterns avances

### Flatten n-deep

In [ ]:
def flatten(nested):
    """Flatten une structure imbriquee de listes/tuples."""
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)
        else:
            yield item

nested = [1, [2, 3, [4, [5, 6]]], 7, (8, 9)]
print(f"flatten : {list(flatten(nested))}")

# Variante : depth-limited (utile en pratique)
def flatten_depth(nested, depth=1):
    if depth == 0 or not isinstance(nested, (list, tuple)):
        yield nested
    else:
        for item in nested:
            yield from flatten_depth(item, depth - 1) if isinstance(item, (list, tuple)) else [item]

### Decoupage en N parties equilibrees

In [ ]:
def chunks_equal(lst, n_chunks):
    """Decoupe lst en n_chunks parts aussi equilibrees que possible."""
    k, m = divmod(len(lst), n_chunks)
    return [lst[i*k + min(i, m):(i+1)*k + min(i+1, m)] for i in range(n_chunks)]

xs = list(range(13))
for i, chunk in enumerate(chunks_equal(xs, 4)):
    print(f"  part {i+1} ({len(chunk)}) : {chunk}")

# === Chunk by size fixe ===
def chunks_size(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

print(f"\nChunks de 4 :")
for c in chunks_size(list(range(13)), 4):
    print(f"  {c}")

## 6. NumPy arrays — creation, reshape, broadcast, masks

In [ ]:
# === Creation ===
a = np.array([1, 2, 3, 4])
b = np.zeros((3, 4))               # (rows, cols)
c = np.ones(5)
d = np.arange(10)                  # [0..9]
e = np.linspace(0, 1, 5)           # [0, 0.25, 0.5, 0.75, 1]
f = np.random.randn(3, 3)
g = np.eye(3)                      # identite

print(f"a shape={a.shape}, dtype={a.dtype}")
print(f"e = {e}")
print(f"g identity:\n{g}")

In [ ]:
# === Reshape ===
x = np.arange(12)
print(f"x = {x}")
print(f"reshape (3, 4):\n{x.reshape(3, 4)}")
print(f"reshape (-1, 2) (inferred):\n{x.reshape(-1, 2)}")

# Reshape vue (memoire partagee) vs copy
v = x.reshape(3, 4)
v[0, 0] = 99
print(f"\nApres v[0,0]=99 :")
print(f"  v[0,0] = {v[0,0]}")
print(f"  x[0]   = {x[0]}  (modifie via la vue)")
x[0] = 0  # restore

In [ ]:
# === Broadcasting ===
# numpy etend automatiquement les shapes compatibles
a = np.array([[1, 2, 3], [4, 5, 6]])    # shape (2, 3)
b = np.array([10, 20, 30])               # shape (3,)
print(f"a + b :\n{a + b}")               # b broadcast a chaque row

# Outer product via broadcast
x = np.arange(4).reshape(-1, 1)          # shape (4, 1)
y = np.arange(3)                          # shape (3,)
print(f"\nouter product :\n{x * y}")     # (4, 3)

In [ ]:
# === Masks et indexation booleenne ===
a = np.array([1, 2, 3, 4, 5, 6])
mask = a > 3
print(f"mask : {mask}")
print(f"a[mask] : {a[mask]}")
print(f"a[a > 3] : {a[a > 3]}")

# Set values via mask
a[a > 3] = 0
print(f"a apres : {a}")

# Combine masks : & | ~ (PAS and/or/not)
b = np.array([1, 5, 2, 4, 3])
print(f"\nb : {b}")
print(f"b[(b > 1) & (b < 5)] : {b[(b > 1) & (b < 5)]}")

# np.where (ternaire vectorise)
c = np.array([10, 20, 30, 40, 50])
print(f"\nnp.where(c > 25, c, -1) : {np.where(c > 25, c, -1)}")

## 7. Comparatif perf — list vs tuple vs set vs dict

In [ ]:
n = 100_000
xs_list = list(range(n))
xs_tuple = tuple(range(n))
xs_set = set(range(n))
xs_dict = {i: None for i in range(n)}

# Memoire
print(f"=== Memoire ({n:,} items) ===")
print(f"  list  : {sys.getsizeof(xs_list):>10,} bytes")
print(f"  tuple : {sys.getsizeof(xs_tuple):>10,} bytes  (immutable -> plus compact)")
print(f"  set   : {sys.getsizeof(xs_set):>10,} bytes")
print(f"  dict  : {sys.getsizeof(xs_dict):>10,} bytes")

# Lookup
target = n - 1
runs = 10
import timeit
t_list = timeit.timeit(lambda: target in xs_list, number=runs)
t_tuple = timeit.timeit(lambda: target in xs_tuple, number=runs)
t_set = timeit.timeit(lambda: target in xs_set, number=runs)
t_dict = timeit.timeit(lambda: target in xs_dict, number=runs)
print(f"\n=== Lookup (membership, {runs} runs) ===")
print(f"  list  : {t_list*1000:.2f} ms")
print(f"  tuple : {t_tuple*1000:.2f} ms")
print(f"  set   : {t_set*1000:.4f} ms  (O(1))")
print(f"  dict  : {t_dict*1000:.4f} ms  (O(1))")

## References

### Documentation officielle
- Python data structures : https://docs.python.org/3/tutorial/datastructures.html
- `collections` : https://docs.python.org/3/library/collections.html
- NumPy basics : https://numpy.org/doc/stable/user/basics.html
- NumPy broadcasting : https://numpy.org/doc/stable/user/basics.broadcasting.html

### Voir aussi (collection)
- [Structure_Pyhton.ipynb](Structure_Pyhton.ipynb)
- [Structures_DataFrame.ipynb](Structures_DataFrame.ipynb)
- [Structures_Polars.ipynb](Structures_Polars.ipynb)
